# Silver V1 — Google Colab Runner

This notebook reads the **existing 50K Bronze snapshot** from the correct Google Drive account and builds **Silver V1**. It neither downloads Bronze again nor overwrites existing Silver outputs.

**Prerequisites**
1. The GitHub repository is **public**: no `GITHUB_TOKEN`, GitHub login or Colab Secret is needed.
2. In Colab, mount the Google account that contains `vietnamese-multilabel-topic-classification`.
3. Run cells in order. If Silver outputs already exist, the preflight intentionally stops before publishing; do not rerun the publishing step.


In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/Ko4rut/vietnamese-multimodal-moderation.git"
BRANCH = __import__("os").environ.get("VTC_GIT_REF", "main")
PROJECT_DIR = Path("/content/project")

# Clone a public repository; no GitHub token or credential helper is required.
if (PROJECT_DIR / ".git").is_dir():
    dirty = subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "status", "--porcelain"],
        capture_output=True, text=True, check=True,
    )
    if dirty.stdout.strip():
        raise RuntimeError("Local /content/project contains uncommitted changes; resolve them first")
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "fetch", "origin", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(PROJECT_DIR), "checkout", "-B", BRANCH, "FETCH_HEAD"],
        check=True,
    )
else:
    if PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
        raise RuntimeError(f"{PROJECT_DIR} exists and is not a Git repository")
    subprocess.run(
        ["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)],
        check=True,
    )

revision = subprocess.check_output(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"], text=True
).strip()
print(f"Using branch: {BRANCH} @ {revision}")


In [ ]:
import subprocess
import sys
from pathlib import Path

# Editable installation makes CLI and dependencies available to new Python processes.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[data]"],
    cwd=str(PROJECT_DIR),
    check=True,
)

# A notebook kernel may not reload editable-install .pth paths until restarted.
# Explicitly add the src-layout package root for imports in this running kernel.
SRC_DIR = (PROJECT_DIR / "src").resolve()
if not (SRC_DIR / "vietnamese_topic_classification" / "__init__.py").is_file():
    raise FileNotFoundError(f"Python package not found in: {SRC_DIR}")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from vietnamese_topic_classification.data.bronze import BronzeReader
from vietnamese_topic_classification.data.silver import SilverPipeline

print("Package import OK:", SRC_DIR)

from vietnamese_topic_classification.data.colab_lake import resolve_mounted_lake


In [ ]:
from google.colab import drive
from pathlib import Path

MOUNT = "/content/vtsnlp_drive"
drive.mount(MOUNT)

# This Colab session must mount the Drive account containing the 50K Bronze snapshot.
DATA_LAKE_ROOT = resolve_mounted_lake(MOUNT)
MANIFEST_NAME = "source_manifest_b81fcce589_seed42_buf10000_n50000.json"

manifest_path = DATA_LAKE_ROOT / "00_source_metadata" / MANIFEST_NAME
if not manifest_path.is_file():
    raise FileNotFoundError(
        f"Bronze manifest not found: {manifest_path}. "
        "Check that the correct Google account is mounted."
    )

reader = BronzeReader(data_lake_root=DATA_LAKE_ROOT, metadata_name=MANIFEST_NAME)
print("Data Lake:", DATA_LAKE_ROOT)
print("Bronze snapshot:", reader.data_path.name)
print("Expected source rows:", reader.manifest["sample_size"])


In [ ]:
# Use canonical planned paths; verified snapshots are safe to reuse.
from vietnamese_topic_classification.data.silver import SilverReader

pipeline = SilverPipeline(reader)
OUTPUT_FILES = pipeline.planned_paths()
existing = [path for path in OUTPUT_FILES.values() if path.exists()]
legacy_paths = [
    DATA_LAKE_ROOT / "02_silver" / OUTPUT_FILES["silver"].name,
    DATA_LAKE_ROOT / "02_silver" / OUTPUT_FILES["rejected"].name,
    DATA_LAKE_ROOT / "05_reports" / OUTPUT_FILES["report"].name,
]
if any(path.exists() for path in legacy_paths):
    raise FileExistsError("Legacy Silver V1 files exist; do not republish without migrating")
SILVER_ALREADY_EXISTS = False
if existing:
    if len(existing) != len(OUTPUT_FILES):
        raise FileExistsError(f"Partial Silver V1 snapshot: {existing}")
    silver_reader = SilverReader(DATA_LAKE_ROOT, OUTPUT_FILES["report"].name)
    verified = silver_reader.verify()
    if silver_reader.report["source_sha256"] != reader.verify()["sha256"]:
        raise AssertionError("Silver V1 report does not match Bronze source")
    SILVER_ALREADY_EXISTS = True
    print("EXISTING SILVER V1 VERIFIED — will skip build:", verified)
else:
    print("No Silver V1 yet; ready to build.")


In [ ]:
# Run the existing CLI in a new Python process.
# PYTHONPATH makes the src-layout imports explicit for this subprocess, too.
import os

command = [
    sys.executable, str(PROJECT_DIR / "scripts/02_build_silver.py"),
    "--manifest-name", MANIFEST_NAME,
    "--data-lake-root", str(DATA_LAKE_ROOT),
]
env = os.environ.copy()
env["PYTHONPATH"] = os.pathsep.join(
    path for path in [str(SRC_DIR), env.get("PYTHONPATH", "")] if path
)
if SILVER_ALREADY_EXISTS:
    print("SILVER V1 ALREADY VERIFIED — SKIPPING BUILD")
else:
    subprocess.run(command, cwd=str(PROJECT_DIR), env=env, check=True)
    print("SILVER PIPELINE COMPLETED")


In [ ]:
# Validate the outputs without re-running the pipeline.
import hashlib
import json

for artifact in OUTPUT_FILES.values():
    assert artifact.is_file(), f"Missing output: {artifact}"

report = json.loads(OUTPUT_FILES["report"].read_text(encoding="utf-8"))
assert report["source_sha256"] == reader.manifest["sha256"]
assert report["input_rows"] == reader.manifest["sample_size"]
assert report["input_rows"] == report["accepted_rows"] + report["rejected_rows"]

def inspect_jsonl(path):
    digest = hashlib.sha256()
    lines = 0
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
            lines += chunk.count(b"\n")
    return lines, digest.hexdigest()

accepted_count, accepted_sha = inspect_jsonl(OUTPUT_FILES["silver"])
rejected_count, rejected_sha = inspect_jsonl(OUTPUT_FILES["rejected"])

assert accepted_count == report["accepted_rows"]
assert rejected_count == report["rejected_rows"]
assert accepted_sha == report["silver_sha256"]
assert rejected_sha == report["rejected_sha256"]

print("SILVER VALIDATION PASSED")
print("Input rows:", report["input_rows"])
print("Accepted:", accepted_count)
print("Rejected:", rejected_count)
print("Rejection reasons:", report["rejection_reasons"])
print("Accepted domains:", len(report["accepted_domain_counts"]))
print("Silver:", OUTPUT_FILES["silver"])
print("Rejected:", OUTPUT_FILES["rejected"])
print("Report:", OUTPUT_FILES["report"])
